# Combined Dataset Preprocessing

This notebook combines the LIAR and ISOT datasets for fake news classification.

Steps:
1. Load LIAR dataset (train.tsv, valid.tsv, test.tsv)
2. Load ISOT dataset (True.csv, Fake.csv)
3. Standardize text features and labels
4. Split ISOT dataset into train/valid/test
5. Combine datasets while maintaining splits
6. Save preprocessed combined datasets


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os

# Column names for LIAR dataset
LIAR_COLUMN_NAMES = [
    'id',
    'label',
    'statement',
    'subject',
    'speaker',
    'job_title',
    'state_info',
    'party_affiliation',
    'barely_true_counts',
    'false_counts',
    'half_true_counts',
    'mostly_true_counts',
    'pants_on_fire_counts',
    'context'
]


In [3]:
# Load and preprocess LIAR dataset
print("Loading LIAR dataset...")

train_df = pd.read_csv('raw/train.tsv', sep='\t', names=LIAR_COLUMN_NAMES)
valid_df = pd.read_csv('raw/valid.tsv', sep='\t', names=LIAR_COLUMN_NAMES)
test_df = pd.read_csv('raw/test.tsv', sep='\t', names=LIAR_COLUMN_NAMES)

# Convert labels to binary
def create_binary_labels(df):
    label_map = {
        'true': 1,
        'mostly-true': 1,
        'half-true': 0,
        'false': 0,
        'pants-fire': 0,
        'barely-true': 0,
    }
    df_copy = df.copy()
    df_copy['label_binary'] = df_copy['label'].map(label_map)
    df_copy.dropna(subset=['label_binary'], inplace=True)
    df_copy['label_binary'] = df_copy['label_binary'].astype(int)
    df_copy['statement'] = df_copy['statement'].fillna('')
    # Drop the original 'label' column to avoid duplicate column names
    df_copy = df_copy.drop(columns=['label'])
    return df_copy

train_df = create_binary_labels(train_df)
valid_df = create_binary_labels(valid_df)
test_df = create_binary_labels(test_df)

# Standardize column names
train_df = train_df.rename(columns={'statement': 'text', 'label_binary': 'label'})
valid_df = valid_df.rename(columns={'statement': 'text', 'label_binary': 'label'})
test_df = test_df.rename(columns={'statement': 'text', 'label_binary': 'label'})

# Add dataset source identifier
train_df['dataset'] = 'LIAR'
valid_df['dataset'] = 'LIAR'
test_df['dataset'] = 'LIAR'

# Select only relevant columns
train_df = train_df[['text', 'label', 'dataset']]
valid_df = valid_df[['text', 'label', 'dataset']]
test_df = test_df[['text', 'label', 'dataset']]

print(f"LIAR - Train: {train_df.shape}, Valid: {valid_df.shape}, Test: {test_df.shape}")


Loading LIAR dataset...
LIAR - Train: (10240, 3), Valid: (1284, 3), Test: (1267, 3)


In [4]:
# Load and preprocess ISOT dataset
print("\nLoading ISOT dataset...")

true_df = pd.read_csv('raw/True.csv')
fake_df = pd.read_csv('raw/Fake.csv')

# Add labels
true_df['label'] = 1
fake_df['label'] = 0

# Combine true and fake
isot_df = pd.concat([true_df, fake_df], ignore_index=True)

# Combine title and text for better features
isot_df['text'] = isot_df['title'].fillna('') + ' ' + isot_df['text'].fillna('')
isot_df['text'] = isot_df['text'].str.strip()

# Add dataset source identifier
isot_df['dataset'] = 'ISOT'

# Select only relevant columns
isot_df = isot_df[['text', 'label', 'dataset']]

print(f"ISOT - Total: {isot_df.shape}")
print(f"ISOT - Label distribution:\n{isot_df['label'].value_counts()}")



Loading ISOT dataset...
ISOT - Total: (44898, 3)
ISOT - Label distribution:
label
0    23481
1    21417
Name: count, dtype: int64


In [5]:
# Split ISOT dataset into train/valid/test (70/15/15)
print("\nSplitting ISOT dataset...")

# First split: train vs (valid + test)
isot_train, isot_temp = train_test_split(
    isot_df, 
    test_size=0.3, 
    stratify=isot_df['label'], 
    random_state=42
)

# Second split: valid vs test
isot_valid, isot_test = train_test_split(
    isot_temp,
    test_size=0.5,
    stratify=isot_temp['label'],
    random_state=42
)

print(f"ISOT splits - Train: {isot_train.shape}, Valid: {isot_valid.shape}, Test: {isot_test.shape}")



Splitting ISOT dataset...
ISOT splits - Train: (31428, 3), Valid: (6735, 3), Test: (6735, 3)


In [6]:
# Combine datasets
print("\nCombining datasets...")

combined_train = pd.concat([train_df, isot_train], ignore_index=True)
combined_valid = pd.concat([valid_df, isot_valid], ignore_index=True)
combined_test = pd.concat([test_df, isot_test], ignore_index=True)

print(f"\nCombined datasets:")
print(f"Train: {combined_train.shape}")
print(f"Valid: {combined_valid.shape}")
print(f"Test: {combined_test.shape}")

print(f"\nTrain label distribution:\n{combined_train['label'].value_counts()}")
print(f"\nTrain dataset distribution:\n{combined_train['dataset'].value_counts()}")



Combining datasets...

Combined datasets:
Train: (41668, 3)
Valid: (8019, 3)
Test: (8002, 3)

Train label distribution:
label
0    23038
1    18630
Name: count, dtype: int64

Train dataset distribution:
dataset
ISOT    31428
LIAR    10240
Name: count, dtype: int64


In [7]:
# Save preprocessed datasets
output_dir = 'processed'
os.makedirs(output_dir, exist_ok=True)

combined_train.to_csv(f'{output_dir}/train_combined.csv', index=False)
combined_valid.to_csv(f'{output_dir}/valid_combined.csv', index=False)
combined_test.to_csv(f'{output_dir}/test_combined.csv', index=False)

print(f"\nSaved preprocessed datasets to {output_dir}/")
print(f"  - train_combined.csv")
print(f"  - valid_combined.csv")
print(f"  - test_combined.csv")



Saved preprocessed datasets to processed/
  - train_combined.csv
  - valid_combined.csv
  - test_combined.csv
